# Schema Inspector

**Doel:** diagnostiek-tool voor parquet-schema's in een External Volume. Print de
**on-disk** Parquet logical types via `pyarrow.parquet.read_schema`, zodat je een
expliciet Spark `StructType` kunt opbouwen voor de ingest-notebook.

**Wanneer gebruiken:**
- Bij een `[PARQUET_TYPE_ILLEGAL]` error tijdens ingestie (Spark weigert een logical type, bijv. `INT32 TIME(MILLIS,true)`).
- Bij onverwachte schema-mismatches tussen bron en doel.
- Voor het bouwen van `TARGET_SCHEMAS` in `staging/02_ingest_azurestorage.ipynb`.

**Wat het niet doet:** geen data lezen, geen tabellen schrijven — pure inspectie.

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO_DEV", "Catalog")
dbutils.widgets.text("schema", "STAGING_AZURESTORAGE", "Schema")
dbutils.widgets.text("volume", "source", "Volume")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
volume  = dbutils.widgets.get("volume")

volume_path = f"/Volumes/{catalog.lower()}/{schema.lower()}/{volume}"
print(f"Volume path: {volume_path}")

## Stap 1 — Lijst parquet-bestanden in het Volume

In [ ]:
import os

all_files = sorted(
    f for f in os.listdir(volume_path) if f.lower().endswith(".parquet")
)

print(f"Aantal parquet-bestanden: {len(all_files)}")
for f in all_files[:20]:
    print(f"  {f}")
if len(all_files) > 20:
    print(f"  ... ({len(all_files) - 20} meer)")

## Stap 2 — Inspecteer schema per file-pattern

Voor elk uniek bestandsprefix (bijv. `ORDER_HEADER`, `ORDER_DETAIL`) wordt één
sample-bestand geïnspecteerd. De pyarrow-schema toont de **logical types** zoals
ze fysiek in het parquet-bestand staan — dat is wat je nodig hebt om een
expliciet Spark `StructType` te bouwen.

In [ ]:
import re
import pyarrow.parquet as pq

# Group bestanden op leading prefix: tekst tot het eerste '_<digit>' of '.<digit>'.
# Voorbeeld: 'ORDER_HEADER_0_0_0.snappy.parquet' -> 'ORDER_HEADER'
patterns: dict[str, str] = {}
for fname in all_files:
    prefix = re.split(r"[_.]\d", fname, maxsplit=1)[0]
    patterns.setdefault(prefix, fname)  # eerste sample per prefix

print(f"Unieke patterns: {list(patterns.keys())}\n")

for prefix, sample in patterns.items():
    full_path = f"{volume_path}/{sample}"
    print(f"=== {prefix}  (sample: {sample}) ===")
    pa_schema = pq.read_schema(full_path)
    print(pa_schema)
    print()

## Volgende stap — vertaal naar Spark `StructType`

Vertaal de pyarrow logical types hierboven naar Spark `StructField`s in de
`TARGET_SCHEMAS` constant in `staging/02_ingest_azurestorage.ipynb`.

**Type-mapping referentie:**

| pyarrow logical type | Spark `DataType` | Opmerking |
|---|---|---|
| `decimal128(p, s)` | `DecimalType(p, s)` | Direct leesbaar |
| `string` | `StringType()` | Direct leesbaar |
| `double` | `DoubleType()` | Direct leesbaar |
| `int32` / `int64` | `IntegerType()` / `LongType()` | Direct leesbaar |
| `timestamp[ms, tz=UTC]` | `TimestampType()` | Direct leesbaar |
| `time32[ms]` (= `INT32 TIME(MILLIS,true)`) | `IntegerType()` lezen, dan casten | **Niet direct leesbaar** — Spark weigert dit logical type. Lees als `IntegerType` (millis sinds middernacht) en converteer in een aparte stap, bijv. `from_unixtime((col / 1000).cast("long"))` of zelf een tijd-string formatten. |
| `time64[us]` / `time64[ns]` | `LongType()` lezen, dan casten | Idem als `time32`. |

Het `INT32 TIME` patroon is de meest voorkomende oorzaak van `[PARQUET_TYPE_ILLEGAL]`.
Definieer die kolommen in het expliciete schema als `IntegerType` zodat Spark de
raw bytes leest en de logical-type-validatie overslaat.